In [738]:
# Stopping Warnings from my Ram
import os
import warnings
os.environ["LOKY_MAX_CPU_COUNT"] = "2"
os.environ["OMP_NUM_THREADS"] = "2"
warnings.filterwarnings("ignore", category=UserWarning)

In [739]:
# Import Libraries

In [740]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [741]:
# Load Dataset

In [742]:
CSV_PATH = "raw_wholesale_customers.csv"
OUT_PATH = "segmented_wholesale_customers.csv"
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
RANDOM_STATE = 42
K = 2

df = pd.read_csv(CSV_PATH)
print("\n=== INITIAL SNAPSHOT ===")
print(df.head())



=== INITIAL SNAPSHOT ===
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185


In [743]:
# Remove Outliers (IQR Capping Using Loop)

In [744]:
X = df[FEATURES].copy()

def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

for col in FEATURES:
    low, high = iqr_fun(X[col])
    X[col] = X[col].clip(lower=low, upper=high)

df[FEATURES] = X

In [745]:
print("\n=== IQR CAP SUMMARY FOR EACH FEATURE ===")
print(
    f"Fresh             low={low:.2f}  high={high:.2f}  |  after cap min={X['Fresh'].min():.2f}  max={X['Fresh'].max():.2f}")
print(
    f"Milk              low={low:.2f}  high={high:.2f}  |  after cap min={X['Milk'].min():.2f}  max={X['Milk'].max():.2f}")
print(
    f"Grocery           low={low:.2f}  high={high:.2f}  |  after cap min={X['Grocery'].min():.2f}  max={X['Grocery'].max():.2f}")
print(
    f"Frozen            low={low:.2f}  high={high:.2f}  |  after cap min={X['Frozen'].min():.2f}  max={X['Frozen'].max():.2f}")
print(
    f"Detergents_Paper  low={low:.2f}  high={high:.2f}  |  after cap min={X['Detergents_Paper'].min():.2f}  max={X['Detergents_Paper'].max():.2f}")
print(
    f"Delicassen        low={low:.2f}  high={high:.2f}  |  after cap min={X['Delicassen'].min():.2f}  max={X['Delicassen'].max():.2f}")

print("\n=== FEATURES HEAD (after IQR cap) ===")
print(X.head())


=== IQR CAP SUMMARY FOR EACH FEATURE ===
Fresh             low=-1709.75  high=3938.25  |  after cap min=3.00  max=37642.75
Milk              low=-1709.75  high=3938.25  |  after cap min=55.00  max=15676.12
Grocery           low=-1709.75  high=3938.25  |  after cap min=3.00  max=23409.88
Frozen            low=-1709.75  high=3938.25  |  after cap min=25.00  max=7772.25
Detergents_Paper  low=-1709.75  high=3938.25  |  after cap min=3.00  max=9419.88
Delicassen        low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25

=== FEATURES HEAD (after IQR cap) ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


In [746]:
# Scale Features

In [747]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nScaled shape:", X_scaled.shape)



Scaled shape: (440, 6)


In [748]:
# Kmeans

In [749]:
print("\n=== ELBOW METHOD (SSE per k) ===")
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    kmeans_labels = kmeans.fit_predict(X_scaled)
    df["Cluster"] = kmeans_labels
    print(f"k={k} → SSE={kmeans.inertia_:.2f}")


=== ELBOW METHOD (SSE per k) ===
k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


In [750]:
# DBSCAN

In [751]:
print("\n=== TUNING DBSCAN (Dynamic EPS) ===")
for test_eps in np.linspace(0.2, 2.0, 10):
    dbscan = DBSCAN(eps=test_eps, min_samples=5)
    dbscan_labels = dbscan.fit_predict(X_scaled)
    n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = list(dbscan_labels).count(-1)
    print(f"EPS={test_eps:.2f} → {n_clusters} clusters {n_noise} noise points.")


=== TUNING DBSCAN (Dynamic EPS) ===
EPS=0.20 → 0 clusters 440 noise points.
EPS=0.40 → 1 clusters 395 noise points.
EPS=0.60 → 1 clusters 286 noise points.
EPS=0.80 → 3 clusters 192 noise points.
EPS=1.00 → 5 clusters 99 noise points.
EPS=1.20 → 1 clusters 56 noise points.
EPS=1.40 → 1 clusters 32 noise points.
EPS=1.60 → 1 clusters 19 noise points.
EPS=1.80 → 2 clusters 10 noise points.
EPS=2.00 → 1 clusters 4 noise points.


In [752]:
# Evaluating and Comparing Models

In [753]:
print("\n" + "="*40)
print("== CLUSTERING METRICS COMPARISON ==")
print("="*40)

# KMeans Performance
km_sil = silhouette_score(X_scaled, kmeans_labels)
km_dbi = davies_bouldin_score(X_scaled, kmeans_labels)
print(f"K-Means Silhouette Score : {km_sil:.3f} (Closer to +1 is better)")
print(f"K-Means Davies-Bouldin   : {km_dbi:.3f} (Lower is better)")


== CLUSTERING METRICS COMPARISON ==
K-Means Silhouette Score : 0.241 (Closer to +1 is better)
K-Means Davies-Bouldin   : 1.312 (Lower is better)


In [754]:
# DBSCAN Performance
n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

if n_clusters_db > 1:
    non_noise_mask = dbscan_labels != -1
    X_filtered = X_scaled[non_noise_mask]
    labels_filtered = dbscan_labels[non_noise_mask]
    db_sil = silhouette_score(X_filtered, labels_filtered)
    db_dbi = davies_bouldin_score(X_filtered, labels_filtered)
    
    print(f"DBSCAN Silhouette Score  : {db_sil:.3f} (Found {n_clusters_db} clusters)")
    print(f"DBSCAN Davies-Bouldin    : {db_dbi:.3f}")
else:
    print(f"DBSCAN Silhouette Score  : (Only found {n_clusters_db} clusters)")

print(f"DBSCAN Outliers Caught   : {n_noise} points identified as noise.")

DBSCAN Silhouette Score  : (Only found 1 clusters)
DBSCAN Outliers Caught   : 4 points identified as noise.


In [755]:
# Sanity Check

In [756]:
EPS = 0.8
MIN_SAMPLES = 5
dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
dbscan_labels = dbscan.fit_predict(X_scaled)
# df["DBSCAN_Cluster"] = dbscan_labels
n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_db = list(dbscan_labels).count(-1)

print(f"DBSCAN found {n_clusters_db} clusters and {n_noise_db} noise points (at EPS={EPS}).")

DBSCAN found 3 clusters and 192 noise points (at EPS=0.8).


In [757]:
# Comparison Performance (KMeans vs DBSCAN)

In [758]:


import joblib

print("\n" + "="*40)
print("== AUTOMATED MODEL SELECTION & SAVE ==")
print("="*40)

km_score = km_sil
db_score = db_sil if 'db_sil' in locals() else -1

print(f"K-Means Silhouette Score : {km_score:.3f}")
print(f"DBSCAN Silhouette Score  : {db_score:.3f}")
print("-" * 40)

if km_score > db_score:
    print(" Win The: K-Means!")
else:
    print(" WIN The: DBSCAN!")


== AUTOMATED MODEL SELECTION & SAVE ==
K-Means Silhouette Score : 0.241
DBSCAN Silhouette Score  : 0.044
----------------------------------------
 Win The: K-Means!


In [759]:
# Final Saved Labeled Dataset

In [760]:
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved clustered dataset with both predictions → {OUT_PATH}")


Saved clustered dataset with both predictions → segmented_wholesale_customers.csv
